# Task 5 — FaceLift reconstruction in Google Colab

This notebook runs the official FaceLift model on the frontal and profile photos independently. It produces a Gaussian 3D representation, rendered views, and a turntable video for each input.

## Run instructions

1. In Colab, select **Runtime → Change runtime type**. Choose **T4 GPU** (or another GPU) and **Runtime version 2026.07** (Python 3.12).
2. Run all cells from top to bottom. The setup cell patches two dependency issues observed with Colab's Python 3.12 runtime: it selects a compatible scikit-image wheel and disables pip build isolation for the CUDA rasterizer.
3. At the upload step, select both `Front-view.png` and `Profile-view.png`. Photos stay in Colab's temporary runtime and are not uploaded to GitHub.
4. Wait for both inference cells and their output checks to finish. Download the ZIP before the Colab runtime ends.

The notebook installs ONNX Runtime without the `rembg[cpu]` extra so it does not replace FaceLift's pinned NumPy. It also checks outputs before creating a ZIP, preventing empty archives.


In [1]:
# Fail early if Colab did not attach an NVIDIA GPU.
import shutil
import subprocess

if not shutil.which('nvidia-smi'):
    raise RuntimeError('No NVIDIA GPU is attached. Select Runtime > Change runtime type > T4 GPU, then rerun from this cell.')
gpu_check = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
print(gpu_check.stdout)
if gpu_check.returncode != 0 or 'GPU 0:' not in gpu_check.stdout:
    raise RuntimeError('Colab did not expose a GPU. Select a GPU runtime, then rerun from this cell.')


GPU 0: Tesla T4 (UUID: GPU-96f4dc36-dbc9-ba61-cdb6-e697e374397c)



If `nvidia-smi` reports that no GPU is available, stop here and change the runtime to GPU. FaceLift inference uses CUDA.

In [2]:
# Clone a clean copy of the official project into Colab's temporary filesystem.
%cd /content
from pathlib import Path
import shutil
import subprocess

repo_path = Path('/content/FaceLift')
if repo_path.exists():
    shutil.rmtree(repo_path)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/weijielyu/FaceLift.git', str(repo_path)], check=True)
%cd /content/FaceLift
!git rev-parse HEAD


/content
/content/FaceLift
0a9420c1480db1e22daa6dee192b76a013295992


## Install FaceLift

The upstream script installs the dependencies and compiles the Gaussian rasterizer. Check that the script finishes successfully before continuing. It does not train a model.

In [3]:
# Patch the upstream installer for Colab's Python 3.12/CUDA build environment, then install.
%cd /content/FaceLift
from pathlib import Path
import subprocess
import sys

setup_file = Path('setup_env.sh')
setup = setup_file.read_text()
old_skimage = 'scikit-image==0.21.0'
old_rasterizer = 'pip install git+https://github.com/graphdeco-inria/diff-gaussian-rasterization'
if old_skimage not in setup or old_rasterizer not in setup:
    raise RuntimeError('Upstream setup script changed; inspect setup_env.sh before continuing.')
setup = setup.replace(old_skimage, 'scikit-image==0.24.0')
setup = setup.replace(old_rasterizer, 'pip install --no-build-isolation git+https://github.com/graphdeco-inria/diff-gaussian-rasterization')
setup_file.write_text(setup)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'ninja'], check=True)
subprocess.run(['bash', str(setup_file)], check=True)


/content/FaceLift


CompletedProcess(args=['bash', 'setup_env.sh'], returncode=0)

## Add the background-removal runtime

FaceLift imports `rembg`, whose background-removal backend is optional. Install ONNX Runtime directly so pip does not upgrade FaceLift's pinned NumPy/Pillow stack. Background removal can use CPU while reconstruction uses the T4 GPU.


In [4]:
import subprocess
import sys

# Install only the backend wheel; its dependencies are already in the FaceLift environment.
# --no-deps prevents pip from replacing NumPy 1.26.4.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'onnxruntime==1.32.0'], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', 'onnxruntime==1.32.0'], returncode=0)

In [12]:
import subprocess, sys
subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"])
!pip install --no-deps --force-reinstall numpy==1.26.4

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


In [1]:
# Confirm required packages and CUDA GPU before upload/inference.
import platform
import subprocess
import numpy as np
import onnxruntime as ort
import torch

print('Python:', platform.python_version())
print('NumPy:', np.__version__)
print('PyTorch:', torch.__version__)
print('PyTorch CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('ONNX Runtime providers:', ort.get_available_providers())
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot access the GPU. Select a GPU runtime and rerun all cells.')
if np.__version__ != '1.26.4':
    raise RuntimeError(f'Expected FaceLift NumPy 1.26.4, found {np.__version__}; stop before inference.')
if 'CPUExecutionProvider' not in ort.get_available_providers():
    raise RuntimeError('ONNX Runtime CPU backend is missing.')
print('GPU:', torch.cuda.get_device_name(0))
commit = subprocess.check_output(['git', '-C', '/content/FaceLift', 'rev-parse', 'HEAD'], text=True).strip()
print('FaceLift commit:', commit)


Python: 3.12.13
NumPy: 1.26.4
PyTorch: 2.4.0+cu124
PyTorch CUDA runtime: 12.4
CUDA available: True
ONNX Runtime providers: ['AzureExecutionProvider', 'CPUExecutionProvider']
GPU: Tesla T4
FaceLift commit: 0a9420c1480db1e22daa6dee192b76a013295992


## Upload both original photographs

Choose both original files in the file dialog. Their names should include `front`/`frontal` and `profile`. They are copied into separate runtime folders and are not committed to GitHub.


In [2]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No files were selected.')
input_root = Path('/content/FaceLift/inputs')
targets = {'frontal': None, 'profile': None}
for uploaded_name, file_bytes in uploaded.items():
    name = uploaded_name.lower()
    if 'profile' in name:
        role = 'profile'
    elif 'front' in name or 'frontal' in name:
        role = 'frontal'
    else:
        continue
    suffix = Path(uploaded_name).suffix.lower() or '.png'
    destination = input_root / role / f'{role}{suffix}'
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(file_bytes)
    targets[role] = destination
missing = [role for role, p in targets.items() if p is None]
if missing:
    raise ValueError(f'Could not identify {missing}. Upload both files with front/profile in their names.')
print('Frontal input:', targets['frontal'])
print('Profile input:', targets['profile'])


Saving profile.png to profile.png
Saving frontal.png to frontal.png
Frontal input: /content/FaceLift/inputs/frontal/frontal.png
Profile input: /content/FaceLift/inputs/profile/profile.png


## Run frontal image


In [4]:
%cd /content/FaceLift
!python inference.py --input_dir /content/FaceLift/inputs/frontal --output_dir /content/FaceLift/outputs/frontal --seed 4 --step_2D 50

/content/FaceLift
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/cupy/__init__.py", line 18, in <module>
    from cupy import _core  # NOQA
    ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/cupy/_core/__init__.py", line 5, in <module>
    from cupy._core import core  # NOQA
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "cupy/_core/core.pyx", line 1, in init cupy._core.core
  File "/usr/local/lib/python3.12/dist-packages/cupy/cuda/__init__.py", line 13, in <module>
    from cupy.cuda import compiler  # NOQA
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/cupy/cuda/compiler.py", line 16, in <module>
    from cupy.cuda import function
  File "cupy/cuda/function.pyx", line 1, in init cupy.cuda.function
  File "cupy/_core/_scalar.pyx", line 1, in init cupy._core._scalar
ImportError: numpy.core.multiarray failed to import (auto-generated because you didn't call 'numpy.import_array()' after cimporting num

In [6]:
%cd /content/FaceLift
!python inference.py --input_dir /content/FaceLift/inputs/frontal --output_dir /content/FaceLift/outputs/frontal --seed 4 --step_2D 50

from pathlib import Path
result_dir = Path('/content/FaceLift/outputs/frontal/frontal')
expected = [result_dir / name for name in ('gaussians.ply', 'output.png', 'turntable.mp4')]
missing = [str(p) for p in expected if not p.is_file() or p.stat().st_size == 0]
if missing:
    raise RuntimeError(f'Missing frontal outputs: {missing}')
print('Frontal outputs verified.')

/content/FaceLift
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/cupy/__init__.py", line 18, in <module>
    from cupy import _core  # NOQA
    ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/cupy/_core/__init__.py", line 5, in <module>
    from cupy._core import core  # NOQA
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "cupy/_core/core.pyx", line 1, in init cupy._core.core
  File "/usr/local/lib/python3.12/dist-packages/cupy/cuda/__init__.py", line 13, in <module>
    from cupy.cuda import compiler  # NOQA
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/cupy/cuda/compiler.py", line 16, in <module>
    from cupy.cuda import function
  File "cupy/cuda/function.pyx", line 1, in init cupy.cuda.function
  File "cupy/_core/_scalar.pyx", line 1, in init cupy._core._scalar
ImportError: numpy.core.multiarray failed to import (auto-generated because you didn't call 'numpy.import_array()' after cimporting num

RuntimeError: Missing frontal outputs: ['/content/FaceLift/outputs/frontal/frontal/gaussians.ply', '/content/FaceLift/outputs/frontal/frontal/output.png', '/content/FaceLift/outputs/frontal/frontal/turntable.mp4']

In [7]:
!python -c "import numpy; print(numpy.__version__)"

1.26.4


In [8]:
from pathlib import Path

p = Path('/content/FaceLift/inference.py')
s = p.read_text()
s = s.replace('from rembg import remove\n', '', 1)
call = '            input_image = remove(input_image)'
if call not in s:
    raise RuntimeError('Could not find FaceLift rembg fallback to patch.')
s = s.replace(call, '            from rembg import remove\n' + call, 1)
p.write_text(s)
print('Patched.')

Patched.


In [10]:
from pathlib import Path

p = Path('/content/FaceLift/utils_folder/face_utils.py')
s = p.read_text().replace('from rembg import remove\n', '', 1)

start = s.index('def prepare_foreground_with_rembg(')
end = s.index('def preprocess_image(', start)

replacement = """def prepare_foreground_with_rembg(input_image_array: np.ndarray) -> np.ndarray:
    # Keep the original image; avoid Colab's incompatible CuPy backend.
    if input_image_array.shape[2] == 4:
        return input_image_array
    if input_image_array.shape[2] == 3:
        height, width = input_image_array.shape[:2]
        rgba = np.zeros((height, width, 4), dtype=np.uint8)
        rgba[:, :, :3] = input_image_array
        rgba[:, :, 3] = 255
        return rgba
    return input_image_array
"""
p.write_text(s[:start] + replacement + s[end:])
print('Background removal disabled; face crop remains enabled.')

Background removal disabled; face crop remains enabled.


## Create labelled front and side screenshots

This extracts a front frame and quarter-turn frame from each generated turntable video.


In [14]:
%cd /content/FaceLift
!python inference.py --input_dir /content/FaceLift/inputs/frontal --output_dir /content/FaceLift/outputs/frontal --seed 4 --step_2D 50

/content/FaceLift
2026-11-21 15:00:15.517622: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
/content/Fac

In [15]:
%cd /content/FaceLift
!python inference.py --input_dir /content/FaceLift/inputs/profile --output_dir /content/FaceLift/outputs/profile --seed 4 --step_2D 50

/content/FaceLift
2026-11-21 15:08:48.716295: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
/content/Fac

In [17]:
!find /content/FaceLift/outputs -type f -print

/content/FaceLift/outputs/frontal/frontal/input.png
/content/FaceLift/outputs/frontal/frontal/multiview.png
/content/FaceLift/outputs/profile/profile/input.png
/content/FaceLift/outputs/profile/profile/multiview.png


In [19]:
%cd /content/FaceLift
!bash -o pipefail -c 'python inference.py --input_dir /content/FaceLift/inputs/frontal --output_dir /content/FaceLift/outputs/frontal --seed 4 --step_2D 50 2>&1 | tee /content/frontal_inference.log'

/content/FaceLift
2026-11-21 15:14:24.309688: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
/content/Fac

In [21]:
!tail -100 /content/frontal_inference.log

2026-11-21 15:14:24.309688: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
/content/FaceLift/gslrm/model/

In [22]:
from pathlib import Path

p = Path('/content/FaceLift/gslrm/model/utils_transformer.py')
s = p.read_text()
flash = 'xops.fmha.flash.FwOp, xops.fmha.flash.BwOp'
cutlass = 'xops.fmha.cutlass.FwOp, xops.fmha.cutlass.BwOp'

count = s.count(flash)
if count != 3:
    raise RuntimeError(f'Expected 3 Flash Attention calls to replace; found {count}.')
p.write_text(s.replace(flash, cutlass))
print('T4-compatible CUTLASS attention selected.')

T4-compatible CUTLASS attention selected.


In [23]:
%cd /content/FaceLift
!python inference.py --input_dir /content/FaceLift/inputs/frontal --output_dir /content/FaceLift/outputs/frontal --seed 4 --step_2D 50

/content/FaceLift
2026-11-21 15:21:15.216747: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
/content/Fac

In [24]:
%cd /content/FaceLift
!python inference.py --input_dir /content/FaceLift/inputs/profile --output_dir /content/FaceLift/outputs/profile --seed 4 --step_2D 50

/content/FaceLift
2026-11-21 15:24:17.145156: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
/content/Fac

In [25]:
import cv2
from PIL import Image, ImageDraw
from pathlib import Path

root = Path('/content/FaceLift')
screenshots = root / 'outputs' / 'screenshots'
screenshots.mkdir(parents=True, exist_ok=True)

def extract_labelled_views(role):
    video_path = root / 'outputs' / role / role / 'turntable.mp4'
    if not video_path.is_file():
        raise FileNotFoundError(f'Missing successful {role} video: {video_path}')
    video = cv2.VideoCapture(str(video_path))
    count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    if count < 2:
        video.release()
        raise RuntimeError(f'Not enough frames in {video_path}')
    for view_name, frame_index in [('front', 0), ('side', count // 4)]:
        video.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
        ok, frame = video.read()
        if not ok:
            video.release()
            raise RuntimeError(f'Could not read {view_name} frame from {video_path}')
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        draw = ImageDraw.Draw(image)
        draw.rectangle((0, 0, image.width, 36), fill=(0, 0, 0))
        draw.text((10, 10), f'{role.capitalize()} input - {view_name} view', fill='white')
        image.save(screenshots / f'{role}_input_{view_name}.png')
    video.release()

extract_labelled_views('frontal')
extract_labelled_views('profile')
print('Screenshots:', *sorted(str(q) for q in screenshots.glob('*.png')), sep='\n')


Screenshots:
/content/FaceLift/outputs/screenshots/frontal_input_front.png
/content/FaceLift/outputs/screenshots/frontal_input_side.png
/content/FaceLift/outputs/screenshots/profile_input_front.png
/content/FaceLift/outputs/screenshots/profile_input_side.png


## Package and download verified results

The archive cell stops if either run lacks any required output, instead of downloading an empty ZIP.


In [26]:
from pathlib import Path
import zipfile
from google.colab import files

output_root = Path('/content/FaceLift/outputs')
required = [output_root / role / role / name for role in ('frontal', 'profile') for name in ('gaussians.ply', 'output.png', 'turntable.mp4')]
missing = [str(p) for p in required if not p.is_file() or p.stat().st_size == 0]
if missing:
    raise RuntimeError(f'Refusing to create an incomplete results ZIP. Missing: {missing}')
zip_path = Path('/content/FaceLift_Task5_results.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for p in output_root.rglob('*'):
        if p.is_file():
            archive.write(p, p.relative_to(output_root))
with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()
if not names:
    raise RuntimeError('Results archive is empty; inference outputs were not found.')
print('Created ZIP:', zip_path, 'files:', len(names), 'size MiB:', round(zip_path.stat().st_size / (1024**2), 2))
files.download(str(zip_path))


Created ZIP: /content/FaceLift_Task5_results.zip files: 14 size MiB: 47.32


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>